# Machine Learning using Sentinel 2 images through OpenEO and Forest Inventory data

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import openeo
from openeo.api.process import Parameter
# OpenEO UDP parameter management system
from openeo_udp import ParameterManager
import geopandas
import json

## Load NFI labels and adjust coordinate system

In [72]:
nfi_labels = geopandas.read_file("nfi_labels.gpkg", layer="nfi")
# reproject to match the cube's CRS 
nfi_labels = nfi_labels.to_crs("EPSG:4326")  

nfi_labels.head()

,label,geometry
0,Eucalyptus,MULTIPOINT ((-8.20735 40.95828))
1,Eucalyptus,MULTIPOINT ((-8.20141 40.96279))
2,Eucalyptus,MULTIPOINT ((-8.19546 40.95379))
3,Eucalyptus,MULTIPOINT ((-8.19547 40.95829))
4,Eucalyptus,MULTIPOINT ((-8.18953 40.95829))


In [73]:
# Drop non forest and other forest
nfi_labels = nfi_labels[nfi_labels["label"]!="Other Forest"]
nfi_labels = nfi_labels[nfi_labels["label"]!="Non forest"]
nfi_labels['label'].value_counts()

label
Shrubland         295
Maritime Pines    157
Eucalyptus         92
Name: count, dtype: int64

## Connect to OpenEO and download satellite image

In [74]:
connection = openeo.connect(
    url="https://openeo.dataspace.copernicus.eu/"
).authenticate_oidc()

time_frame = ["2024-01-01,2024-12-30"]
time_post= ["2024-09-30", "2024-10-10"] # + last
time_pre=  ["2024-08-20", "2024-09-01"]  # + last

current_params = {
        "location_name": "Reriz e Gafanhao, North Portugal",
        "bounding_box": Parameter(
            "bounding_box",
            description="Fire area in 2024",
            default= {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}
),
        "time": Parameter(
            "time",
            description="Temporal range for data acquisition",
            # default=["2024-09-30", "2024-10-30"], # + last
            default=["2024-08-01", "2024-09-01"],  # + last
        ),
        "bands": Parameter(
            "bands",
            description="Sentinel-2 bands required for APA calculation",
            default=  ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
#,
        ),
        "collection": Parameter(
            "collection",
            description="Data collection identifier",
            default="SENTINEL2_L2A",
        ),
        "cloud_cover": Parameter(
            "cloud_cover",
            description="Maximum cloud cover percentage",
            default=30,
        ),
    }


Authenticated using refresh token.


In [75]:
def load_and_sample(params, time, res=100):
    """
    Input: Set of parameters, time frame, desired resolution
    Output: Cube that contains image
    """ 
    s2cube = connection.load_collection(
    params["collection"].default,
    temporal_extent=time,
    spatial_extent=params["bounding_box"].default,
    bands=params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= params["cloud_cover"].default,
    },
    )
    # s2cube = s2cube.reduce_dimension(dimension="t", reducer="last")
    s2cube = s2cube.resample_spatial(
        resolution=[res, res],
      method="near",
    )

    return s2cube



# def calculate_ndvi(data):
#      """ 
#      Input: Cube with bands 04 and 8
#      Output: NBR
#      """
#      B04, B08 = (
#           data[2],
#           data[5]
#           )
#      ndvi = (B08 - B04) / (B08 + B04)
#      return ndvi

## Generate Features from S2 image

In [ ]:
# Define Resolution
res = 20
# Load Satellite image (for now post image)
s2cube = load_and_sample(current_params, time_post, res)
#Define features from cube
features = s2cube.aggregate_spatial

In [ ]:
#Ensure right geometry type of labels
nfi_labels["geometry"] = nfi_labels["geometry"].apply(lambda g: g.geoms[0] if g.geom_type == "MultiPoint" else g)

#Convert to Geojson
nfi_labels_gj = nfi_labels[["label", "geometry"]]
geojson = json.loads(nfi_labels_gj.to_json())

In [79]:
# Convert image to vector data
features = s2cube.aggregate_spatial(
    geometries=geojson,
    reducer="mean",)

In [ ]:
# # Download the features as a JSON file 
# features.download("features.json", format="JSON")

# # Open them into a variable
# with open("features.json") as f:
#     raw = json.load(f)

In [81]:
job = features.create_job(out_format="JSON")
job.start_and_wait()
job.get_results().download_files("features_output")

0:00:00 Job 'j-2606231225074c3b8593f1e2d4a86c39': send 'start'
0:00:02 Job 'j-2606231225074c3b8593f1e2d4a86c39': queued (progress 0%)
0:00:08 Job 'j-2606231225074c3b8593f1e2d4a86c39': queued (progress 0%)
0:00:14 Job 'j-2606231225074c3b8593f1e2d4a86c39': queued (progress 0%)
0:00:22 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:00:33 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:00:46 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:01:02 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:01:21 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:01:50 Job 'j-2606231225074c3b8593f1e2d4a86c39': running (progress N/A)
0:02:20 Job 'j-2606231225074c3b8593f1e2d4a86c39': finished (progress 100%)


[WindowsPath('features_output/timeseries.json'),
 WindowsPath('features_output/job-results.json')]

In [84]:
with open("features_output/timeseries.json") as f:
    raw = json.load(f)

In [85]:
# Inspect the raw data structure 
print(raw if isinstance(raw, dict) and len(raw) < 5 else list(raw.items())[:2] if isinstance(raw, dict) else raw[:2])


{'2024-09-30T00:00:00Z': [[270.0, 347.0, 495.0, 665.0, 1103.0, 1270.0, 1358.0, 1158.0, 812.0], [245.0, 305.0, 422.0, 681.0, 1076.0, 1207.0, 1264.0, 1305.0, 911.0], [381.0, 455.0, 583.0, 784.0, 1067.0, 1057.0, 1165.0, 1895.0, 1696.0], [192.0, 186.0, 260.0, 329.0, 389.0, 414.0, 508.0, 1196.0, 1339.0], [249.0, 275.0, 404.0, 582.0, 781.0, 807.0, 1004.0, 1127.0, 868.0], [195.0, 196.0, 277.0, 497.0, 800.0, 686.0, 1125.0, 1068.0, 679.0], [315.0, 411.0, 591.0, 686.0, 1257.0, 1706.0, 1491.0, 1285.0, 820.0], [274.0, 367.0, 525.0, 626.0, 1155.0, 1191.0, 1319.0, 1155.0, 774.0], [261.0, 338.0, 479.0, 663.0, 995.0, 1102.0, 1288.0, 1335.0, 976.0], [264.0, 309.0, 429.0, 449.0, 508.0, 759.0, 663.0, 1131.0, 1077.0], [191.0, 192.0, 269.0, 382.0, 497.0, 469.0, 529.0, 1342.0, 1494.0], [258.0, 258.0, 335.0, 383.0, 505.0, 553.0, 570.0, 1370.0, 1566.0], [325.0, 394.0, 583.0, 721.0, 1174.0, 1434.0, 1595.0, 1309.0, 825.0], [242.0, 305.0, 442.0, 572.0, 910.0, 1021.0, 1091.0, 1309.0, 1165.0], [261.0, 342.0, 480.0

## Machine Learning

In [86]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# flatten remaining timestamps into one feature vector per point
timestamps = list(raw.keys())
n_points = len(nfi_labels)

X = []
for i in range(n_points):
    row_features = []
    for ts in timestamps:
        row_features.extend(raw[ts][i])
    X.append(row_features)

X = np.array(X, dtype=float)
y = nfi_labels["label"].values

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# train RF
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

    Eucalyptus       0.46      0.33      0.39        18
Maritime Pines       0.61      0.62      0.62        32
     Shrubland       0.78      0.83      0.80        59

      accuracy                           0.69       109
     macro avg       0.62      0.60      0.60       109
  weighted avg       0.68      0.69      0.68       109

